#Baseline com 0.30 de amostra



In [2]:
import tensorflow as tf
import pandas as pd
from tensorflow.keras.layers import TextVectorization
from tensorflow.keras.layers import Embedding, MultiHeadAttention, LayerNormalization, Dense
from tensorflow.keras.models import Model

data = pd.read_csv('../data/df_traduzido.csv').sample(frac=0.3)
data.head(3)

,Time_submitted,Review,Rating,Total_thumbsup,Reply,Review_Traduzida,Review_trazudida
5090,2022-06-28 11:52:08,7 ads in one go that's too much,1,0,NaN,NaN,"7 anúncios de uma vez só, isso é demais"
5386,2022-06-27 19:16:03,Love it easy to use...great selection of songs,4,0,NaN,NaN,"Adorei, fácil de usar...ótima seleção de músicas"
57328,2022-01-29 07:29:42,My app keeps crashing whenever I exit from the...,1,189,NaN,NaN,Meu aplicativo continua travando sempre que sa...


In [ ]:
data.drop(['Reply', 'Review_Traduzida'], axis=1, inplace=True)


In [ ]:
data.rename(columns={'Review_trazudida': 'Review_Traduzida'}, inplace=True)

In [ ]:
data.head(5)

,Time_submitted,Review,Rating,Total_thumbsup,Review_Traduzida
22713,2022-05-13 07:57:47,Nice app has many genres and can make your own...,5,0,O aplicativo legal tem muitos gêneros e você p...
50687,2022-02-21 11:01:11,I absolutely love Spotify it is great music Pl...,5,0,"Eu amo muito o Spotify, é uma ótima música. To..."
14827,2022-06-04 20:23:28,There are many things that the other apps lead...,4,18,Há muitas coisas que os outros aplicativos lid...
14149,2022-06-06 18:25:54,i can't play music on my samsung galaxy A10 al...,1,1,não consigo tocar música no meu samsung galaxy...
52928,2022-02-12 13:40:05,User interface nests too much. Hard to manage ...,1,0,A interface do usuário aninha muito. Difícil g...


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

class TransformerEncoder(layers.Layer):
    def __init__(self, embed_dim, dense_dim, num_heads, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.dense_dim = dense_dim
        self.num_heads = num_heads
        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim)
        self.dense_proj = keras.Sequential(
            [layers.Dense(dense_dim, activation="relu"),
             layers.Dense(embed_dim),]
        )
        self.layernorm_1 = layers.LayerNormalization()
        self.layernorm_2 = layers.LayerNormalization()

    def call(self, inputs, mask=None):
        if mask is not None:
            mask = mask[:, tf.newaxis, :]
        attention_output = self.attention(
            inputs, inputs, attention_mask=mask)
        proj_input = self.layernorm_1(inputs + attention_output)
        proj_output = self.dense_proj(proj_input)
        return self.layernorm_2(proj_input + proj_output)

    def get_config(self):
        config = super().get_config()
        config.update({
            "embed_dim": self.embed_dim,
            "num_heads": self.num_heads,
            "dense_dim": self.dense_dim,
        })
        return config



In [ ]:
print(data.columns)

Index(['Time_submitted', 'Review', 'Rating', 'Total_thumbsup',
       'Review_Traduzida'],
      dtype='object')


In [ ]:
text_pairs = []

for line in data.itertuples(index=False):
    english = str(line.Review)  # Converter para string
    portugues = str(line.Review_Traduzida)  # Converter para string

    # Tratar valores ausentes ou NaN, substituindo por uma string vazia
    if pd.isna(portugues):
        portugues = ""

    portugues = "[start] " + portugues + " [end]"

    text_pairs.append((english, portugues))


In [ ]:
import random
print(random.choice(text_pairs))

random.shuffle(text_pairs)

num_val_samples = int(0.15 * len(text_pairs))
num_train_samples = len(text_pairs) - 2 * num_val_samples

train_pairs = text_pairs[:num_train_samples]
val_pairs = text_pairs[num_train_samples:num_train_samples + num_val_samples]
test_pairs = text_pairs[num_train_samples + num_val_samples:]

('Good. But it can only connect to my boom box for short time.', '[start] Bom. Mas ele só consegue se conectar ao meu aparelho de som por um curto período. [end]')


In [ ]:
type(train_pairs)

list

In [ ]:
import string
import re

strip_chars = string.punctuation + "¿"
strip_chars = strip_chars.replace("[", "")
strip_chars = strip_chars.replace("]", "")

def custom_standardization(input_string):
    lowercase = tf.strings.lower(input_string)
    return tf.strings.regex_replace(
        lowercase, f"[{re.escape(strip_chars)}]", "")

vocab_size = 15000
sequence_length = 30

source_vectorization = layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length,
)
target_vectorization = layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length + 1,
    standardize=custom_standardization,
)

train_english_texts = [pair[0] for pair in train_pairs]
train_spanish_texts = [pair[1] for pair in train_pairs]
source_vectorization.adapt(train_english_texts)
target_vectorization.adapt(train_spanish_texts)

In [ ]:
batch_size = 64

def format_dataset(eng, pt):
    eng = source_vectorization(eng)
    pt = target_vectorization(pt)
    return ({
        "english": eng,
        "portugues": pt[:, :-1],
    }, pt[:, 1:])

def make_dataset(pairs):
    eng_texts, pt_texts = zip(*pairs)
    eng_texts = list(eng_texts)
    pt_texts = list(pt_texts)
    dataset = tf.data.Dataset.from_tensor_slices((eng_texts, pt_texts))
    dataset = dataset.batch(batch_size)
    dataset = dataset.map(format_dataset, num_parallel_calls=4)
    return dataset.shuffle(2048).prefetch(16).cache()

train_ds = make_dataset(train_pairs)
val_ds = make_dataset(val_pairs)

In [ ]:
for inputs, targets in train_ds.take(1):
    print(f"inputs['english'].shape: {inputs['english'].shape}")
    print(f"inputs['portugues'].shape: {inputs['portugues'].shape}")
    print(f"targets.shape: {targets.shape}")



inputs['english'].shape: (64, 30)
inputs['portugues'].shape: (64, 30)
targets.shape: (64, 30)


In [ ]:
class TransformerDecoder(layers.Layer):
    def __init__(self, embed_dim, dense_dim, num_heads, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.dense_dim = dense_dim
        self.num_heads = num_heads
        self.attention_1 = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim)
        self.attention_2 = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim)
        self.dense_proj = keras.Sequential(
            [layers.Dense(dense_dim, activation="relu"),
             layers.Dense(embed_dim),]
        )
        self.layernorm_1 = layers.LayerNormalization()
        self.layernorm_2 = layers.LayerNormalization()
        self.layernorm_3 = layers.LayerNormalization()
        self.supports_masking = True

    def get_causal_attention_mask(self, inputs):
        input_shape = tf.shape(inputs)
        batch_size, sequence_length = input_shape[0], input_shape[1]
        i = tf.range(sequence_length)[:, tf.newaxis]
        j = tf.range(sequence_length)
        mask = tf.cast(i >= j, dtype="int32")
        mask = tf.reshape(mask, (1, input_shape[1], input_shape[1]))
        mult = tf.concat(
            [tf.expand_dims(batch_size, -1),
             tf.constant([1, 1], dtype=tf.int32)], axis=0)
        return tf.tile(mask, mult)

    def call(self, inputs, encoder_outputs, mask=None):
        causal_mask = self.get_causal_attention_mask(inputs)
        if mask is not None:
            padding_mask = tf.cast(
                mask[:, tf.newaxis, :], dtype="int32")
            padding_mask = tf.minimum(padding_mask, causal_mask)
        else:
            padding_mask = mask
        attention_output_1 = self.attention_1(
            query=inputs,
            value=inputs,
            key=inputs,
            attention_mask=causal_mask)
        attention_output_1 = self.layernorm_1(inputs + attention_output_1)
        attention_output_2 = self.attention_2(
            query=attention_output_1,
            value=encoder_outputs,
            key=encoder_outputs,
            attention_mask=padding_mask,
        )
        attention_output_2 = self.layernorm_2(
            attention_output_1 + attention_output_2)
        proj_output = self.dense_proj(attention_output_2)
        return self.layernorm_3(attention_output_2 + proj_output)

class PositionalEmbedding(layers.Layer):
    def __init__(self, sequence_length, input_dim, output_dim, **kwargs):
        super().__init__(**kwargs)
        self.token_embeddings = layers.Embedding(
            input_dim=input_dim, output_dim=output_dim)
        self.position_embeddings = layers.Embedding(
            input_dim=sequence_length, output_dim=output_dim)
        self.sequence_length = sequence_length
        self.input_dim = input_dim
        self.output_dim = output_dim

    def call(self, inputs):
        length = tf.shape(inputs)[-1]
        positions = tf.range(start=0, limit=length, delta=1)
        embedded_tokens = self.token_embeddings(inputs)
        embedded_positions = self.position_embeddings(positions)
        return embedded_tokens + embedded_positions

    def get_config(self):
        config = super(PositionalEmbedding, self).get_config()
        config.update({
            "output_dim": self.output_dim,
            "sequence_length": self.sequence_length,
            "input_dim": self.input_dim,
        })
        return config

In [ ]:
embed_dim = 256
dense_dim = 2048
num_heads = 6

encoder_inputs = keras.Input(shape=(None,), dtype="int64", name="english")
x = PositionalEmbedding(sequence_length, vocab_size, embed_dim)(encoder_inputs)
encoder_outputs = TransformerEncoder(embed_dim, dense_dim, num_heads)(x)

decoder_inputs = keras.Input(shape=(None,), dtype="int64", name="portugues")
x = PositionalEmbedding(sequence_length, vocab_size, embed_dim)(decoder_inputs)
x = TransformerDecoder(embed_dim, dense_dim, num_heads)(x, encoder_outputs)
x = layers.Dropout(0.5)(x)
decoder_outputs = layers.Dense(vocab_size, activation="softmax")(x)
transformer = keras.Model([encoder_inputs, decoder_inputs], decoder_outputs)

transformer.compile(
    optimizer="rmsprop",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"])

with tf.device('/device:GPU:0'):
    transformer.fit(train_ds, epochs=30, validation_data=val_ds)

Epoch 1/30
203/203 ━━━━━━━━━━━━━━━━━━━━ 43s 137ms/step - accuracy: 0.3097 - loss: 5.2378 - val_accuracy: 0.3961 - val_loss: 3.6628
Epoch 2/30
203/203 ━━━━━━━━━━━━━━━━━━━━ 16s 65ms/step - accuracy: 0.4046 - loss: 3.6025 - val_accuracy: 0.4353 - val_loss: 3.3440
Epoch 3/30
203/203 ━━━━━━━━━━━━━━━━━━━━ 13s 66ms/step - accuracy: 0.4623 - loss: 3.1923 - val_accuracy: 0.5406 - val_loss: 2.7162
Epoch 4/30
203/203 ━━━━━━━━━━━━━━━━━━━━ 21s 67ms/step - accuracy: 0.5669 - loss: 2.5618 - val_accuracy: 0.6034 - val_loss: 2.3348
Epoch 5/30
203/203 ━━━━━━━━━━━━━━━━━━━━ 20s 67ms/step - accuracy: 0.6206 - loss: 2.1905 - val_accuracy: 0.6375 - val_loss: 2.1183
Epoch 6/30
203/203 ━━━━━━━━━━━━━━━━━━━━ 21s 67ms/step - accuracy: 0.6549 - loss: 1.9393 - val_accuracy: 0.6505 - val_loss: 2.0227
Epoch 7/30
203/203 ━━━━━━━━━━━━━━━━━━━━ 14s 68ms/step - accuracy: 0.6823 - loss: 1.7418 - val_accuracy: 0.6570 - val_loss: 1.9906
Epoch 8/30
203/203 ━━━━━━━━━━━━━━━━━━━━ 20s 65ms/step - accuracy: 0.7032 - loss: 1.5954 -

In [ ]:
# Salvando para uso posterior
transformer.save('../modelos/20240903_modelo_transformer_tradutor_50epochs.keras')

In [ ]:
import numpy as np

pt_vocab = target_vectorization.get_vocabulary()
pt_index_lookup = dict(zip(range(len(pt_vocab)), pt_vocab))
max_decoded_sentence_length = 30

def decode_sequence(input_sentence):
    tokenized_input_sentence = source_vectorization([input_sentence])
    decoded_sentence = "[start]"

    for i in range(max_decoded_sentence_length):
        tokenized_target_sentence = target_vectorization(
            [decoded_sentence])[:, :-1]
        predictions = transformer([tokenized_input_sentence, tokenized_target_sentence])
        sampled_token_index = np.argmax(predictions[0, i, :])
        sampled_token = pt_index_lookup[sampled_token_index]
        decoded_sentence += " " + sampled_token

        if sampled_token == "[end]":
            break

    return decoded_sentence

In [ ]:
test_eng_texts = [pair[0] for pair in test_pairs]
for _ in range(30):
    input_sentence = random.choice(test_eng_texts)
    print("-")
    print(input_sentence)
    print(decode_sequence(input_sentence))

-
Excellent 👌... Good for burning calories....
[start] excelente 👌 bom para cima de gravar [end]
-
After having a premium family account for years, we're starting to be fed up with new issues. Recently, I lost the option to play certain Playlist at random or not. I can shuffle 2 of the 6 Playlist I made myself but not the others. At 16$ monthly, that's annoying.
[start] depois de ter uma conta familiar há anos passados para começar a usar problemas com a nova recentemente perdi a opção de reprodução certas playlists ou na sua playlist ou
-
I really like the app. However, the fact that you have to pay to listen to specific songs just bugs me. My parents won't pay for it, but some of the songs I like, such as Savage and Heart Attack, I have to go through playlists or find the song, put it into one of my playlists, and even then, I might not get to listen to it! So if you could change that, it would absolutely get you a 5 star rating, but until you change that, no way. Edit: Also, I agree

In [ ]:
#comentarios:
# REVIEW INVENTADA: 'Spotify has an amazing music selection, but the constant ads and limited skips make it frustrating to use without premium'
# REVIEW INVENTADA: 'Great audio quality, but I keep losing control over the volume when casting to my smart speaker. Is really inconvenient!'
# FRASE ALEATÓRIA QUE NÃO TEM SENTIDO : 'I love to eat Pizza, when is cold I drink wine'
# FRASE DA BASE TREINO: 'Unable to login .After this new update it shows error everytime I try to login

In [ ]:
reviews = ['Spotify has an amazing music selection, but the constant ads and limited skips make it frustrating to use without premium',
           'Great audio quality, but I keep losing control over the volume when casting to my smart speaker. Is really inconvenient!',
           'I love to eat pasta, when is cold I drink wine.',
           'Unable to login .After this new update it shows error everytime I try to login']

# Aplique a função decode_sequence para cada frase
for review in reviews:
    result = decode_sequence(review)
    print(result)

[start] o spotify tem uma seleção musical incrível mas os anúncios constantes e pulos limitados É limitado de usar o premium [end]
[start] Ótima qualidade de áudio mas continuo perdendo o controle ao volume ao transmitir para altofalante inteligente o meu altofalante realmente irritante [end]
[start] eu adoro o spoyify o ritmo é surreal quando o ritmo [end]
[start] não é possível fazer login depois dessa nova atualização ele mostra erro toda vez que tento fazer login [end]


#Modelo com ajustes - 60% de amostra
####com 0.60 de amostra, (para melhorar a generalização)
####com aumento do voca_size para 30mil (cobrir mais palavras e expressões)
####com sequence_length = 40 (capturar mais contexto em frases maiores)
####com num_heads = 12 (capturar melhor a relação entre as palavras)
####com dense_dim = 4096 (para capturar melhor os padrões)
####com layers.Dropout(0.7) (forçar melhor a generalização)
####com epochs=50 (melhorar o desempenho)
####com optimizer="adam" (mais estavel para modelos transformer)

Caso não funcione, próximo passo aumentar o embed_dim = 512
e adicionar o early_stop


In [5]:
import tensorflow as tf
import pandas as pd
from tensorflow.keras.layers import TextVectorization
from tensorflow.keras.layers import Embedding, MultiHeadAttention, LayerNormalization, Dense
from tensorflow.keras.models import Model

data = pd.read_csv('../data/df_traduzido.csv').sample(frac=0.6)
data.head(3)

,Time_submitted,Review,Rating,Total_thumbsup,Reply,Review_Traduzida,Review_trazudida
40182,2022-04-04 10:25:35,"It's trash without premium, I get not having s...",1,3,NaN,NaN,"É lixo sem premium, eu entendo não ter alguns ..."
7412,2022-06-23 03:11:50,I love this when I'm in a mood I have a Playli...,5,1,NaN,NaN,Eu adoro isso quando estou de mau humor. Tenho...
54514,2022-02-06 23:08:00,Good choices but after going thru all artists ...,3,0,NaN,NaN,"Boas escolhas, mas depois de passar por todos ..."


In [ ]:
data.drop(['Reply', 'Review_Traduzida'], axis=1, inplace=True)


In [ ]:
data.rename(columns={'Review_trazudida': 'Review_Traduzida'}, inplace=True)

In [ ]:
data.head(5)

,Time_submitted,Review,Rating,Total_thumbsup,Review_Traduzida
26082,2022-05-03 18:36:13,I've been using Spotify for Abt as long as I'v...,1,0,"Eu uso o Spotify há muito tempo, desde que com..."
46324,2022-03-08 19:11:23,It's not working at all It's not even liking t...,1,1,Não está funcionando de jeito nenhum. Nem esto...
35219,2022-04-15 01:02:43,Needs a version for chromebook. Using it on a ...,3,0,Precisa de uma versão para Chromebook. Usá-lo ...
36788,2022-04-13 13:31:34,Been having awful trouble lately with Spotify ...,1,0,Tenho tido problemas terríveis ultimamente com...
9628,2022-06-18 04:55:36,Just a W app. Helps with my workout routine 💪,5,0,Apenas um aplicativo W. Ajuda com minha rotina...


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

class TransformerEncoder(layers.Layer):
    def __init__(self, embed_dim, dense_dim, num_heads, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.dense_dim = dense_dim
        self.num_heads = num_heads
        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim)
        self.dense_proj = keras.Sequential(
            [layers.Dense(dense_dim, activation="relu"),
             layers.Dense(embed_dim),]
        )
        self.layernorm_1 = layers.LayerNormalization()
        self.layernorm_2 = layers.LayerNormalization()

    def call(self, inputs, mask=None):
        if mask is not None:
            mask = mask[:, tf.newaxis, :]
        attention_output = self.attention(
            inputs, inputs, attention_mask=mask)
        proj_input = self.layernorm_1(inputs + attention_output)
        proj_output = self.dense_proj(proj_input)
        return self.layernorm_2(proj_input + proj_output)

    def get_config(self):
        config = super().get_config()
        config.update({
            "embed_dim": self.embed_dim,
            "num_heads": self.num_heads,
            "dense_dim": self.dense_dim,
        })
        return config



In [ ]:
print(data.columns)

Index(['Time_submitted', 'Review', 'Rating', 'Total_thumbsup',
       'Review_Traduzida'],
      dtype='object')


In [ ]:
text_pairs = []

for line in data.itertuples(index=False):
    english = str(line.Review)  # Converter para string
    portugues = str(line.Review_Traduzida)  # Converter para string

    # Tratar valores ausentes ou NaN, substituindo por uma string vazia
    if pd.isna(portugues):
        portugues = ""

    portugues = "[start] " + portugues + " [end]"

    text_pairs.append((english, portugues))


In [ ]:
import random
print(random.choice(text_pairs))

random.shuffle(text_pairs)

num_val_samples = int(0.15 * len(text_pairs))
num_train_samples = len(text_pairs) - 2 * num_val_samples

train_pairs = text_pairs[:num_train_samples]
val_pairs = text_pairs[num_train_samples:num_train_samples + num_val_samples]
test_pairs = text_pairs[num_train_samples + num_val_samples:]

('Would give 3 stars, if not for my history with this app. Please get it back to shape. Some design decisions are hard to understand, some technical aspects reminds rather an alpha version than long developed mature application. Let me give you 5 stars back again.', '[start] Daria 3 estrelas, se não fosse pelo meu histórico com este aplicativo. Por favor, coloque-o de volta em forma. Algumas decisões de design são difíceis de entender, alguns aspectos técnicos lembram mais uma versão alfa do que um aplicativo maduro desenvolvido há muito tempo. Deixe-me dar 5 estrelas de volta novamente. [end]')


In [ ]:
type(train_pairs)

list

In [ ]:
import string
import re

strip_chars = string.punctuation + "¿"
strip_chars = strip_chars.replace("[", "")
strip_chars = strip_chars.replace("]", "")

def custom_standardization(input_string):
    lowercase = tf.strings.lower(input_string)
    return tf.strings.regex_replace(
        lowercase, f"[{re.escape(strip_chars)}]", "")

vocab_size = 30000
sequence_length = 40

source_vectorization = layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length,
)
target_vectorization = layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length + 1,
    standardize=custom_standardization,
)

train_english_texts = [pair[0] for pair in train_pairs]
train_spanish_texts = [pair[1] for pair in train_pairs]
source_vectorization.adapt(train_english_texts)
target_vectorization.adapt(train_spanish_texts)

In [ ]:
batch_size = 64

def format_dataset(eng, pt):
    eng = source_vectorization(eng)
    pt = target_vectorization(pt)
    return ({
        "english": eng,
        "portugues": pt[:, :-1],
    }, pt[:, 1:])

def make_dataset(pairs):
    eng_texts, pt_texts = zip(*pairs)
    eng_texts = list(eng_texts)
    pt_texts = list(pt_texts)
    dataset = tf.data.Dataset.from_tensor_slices((eng_texts, pt_texts))
    dataset = dataset.batch(batch_size)
    dataset = dataset.map(format_dataset, num_parallel_calls=4)
    return dataset.shuffle(2048).prefetch(16).cache()

train_ds = make_dataset(train_pairs)
val_ds = make_dataset(val_pairs)

In [ ]:
for inputs, targets in train_ds.take(1):
    print(f"inputs['english'].shape: {inputs['english'].shape}")
    print(f"inputs['portugues'].shape: {inputs['portugues'].shape}")
    print(f"targets.shape: {targets.shape}")



inputs['english'].shape: (64, 40)
inputs['portugues'].shape: (64, 40)
targets.shape: (64, 40)


In [ ]:
class TransformerDecoder(layers.Layer):
    def __init__(self, embed_dim, dense_dim, num_heads, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.dense_dim = dense_dim
        self.num_heads = num_heads
        self.attention_1 = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim)
        self.attention_2 = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim)
        self.dense_proj = keras.Sequential(
            [layers.Dense(dense_dim, activation="relu"),
             layers.Dense(embed_dim),]
        )
        self.layernorm_1 = layers.LayerNormalization()
        self.layernorm_2 = layers.LayerNormalization()
        self.layernorm_3 = layers.LayerNormalization()
        self.supports_masking = True

    def get_causal_attention_mask(self, inputs):
        input_shape = tf.shape(inputs)
        batch_size, sequence_length = input_shape[0], input_shape[1]
        i = tf.range(sequence_length)[:, tf.newaxis]
        j = tf.range(sequence_length)
        mask = tf.cast(i >= j, dtype="int32")
        mask = tf.reshape(mask, (1, input_shape[1], input_shape[1]))
        mult = tf.concat(
            [tf.expand_dims(batch_size, -1),
             tf.constant([1, 1], dtype=tf.int32)], axis=0)
        return tf.tile(mask, mult)

    def call(self, inputs, encoder_outputs, mask=None):
        causal_mask = self.get_causal_attention_mask(inputs)
        if mask is not None:
            padding_mask = tf.cast(
                mask[:, tf.newaxis, :], dtype="int32")
            padding_mask = tf.minimum(padding_mask, causal_mask)
        else:
            padding_mask = mask
        attention_output_1 = self.attention_1(
            query=inputs,
            value=inputs,
            key=inputs,
            attention_mask=causal_mask)
        attention_output_1 = self.layernorm_1(inputs + attention_output_1)
        attention_output_2 = self.attention_2(
            query=attention_output_1,
            value=encoder_outputs,
            key=encoder_outputs,
            attention_mask=padding_mask,
        )
        attention_output_2 = self.layernorm_2(
            attention_output_1 + attention_output_2)
        proj_output = self.dense_proj(attention_output_2)
        return self.layernorm_3(attention_output_2 + proj_output)

class PositionalEmbedding(layers.Layer):
    def __init__(self, sequence_length, input_dim, output_dim, **kwargs):
        super().__init__(**kwargs)
        self.token_embeddings = layers.Embedding(
            input_dim=input_dim, output_dim=output_dim)
        self.position_embeddings = layers.Embedding(
            input_dim=sequence_length, output_dim=output_dim)
        self.sequence_length = sequence_length
        self.input_dim = input_dim
        self.output_dim = output_dim

    def call(self, inputs):
        length = tf.shape(inputs)[-1]
        positions = tf.range(start=0, limit=length, delta=1)
        embedded_tokens = self.token_embeddings(inputs)
        embedded_positions = self.position_embeddings(positions)
        return embedded_tokens + embedded_positions

    def get_config(self):
        config = super(PositionalEmbedding, self).get_config()
        config.update({
            "output_dim": self.output_dim,
            "sequence_length": self.sequence_length,
            "input_dim": self.input_dim,
        })
        return config

In [ ]:
embed_dim = 256
dense_dim = 4096
num_heads = 12

encoder_inputs = keras.Input(shape=(None,), dtype="int64", name="english")
x = PositionalEmbedding(sequence_length, vocab_size, embed_dim)(encoder_inputs)
encoder_outputs = TransformerEncoder(embed_dim, dense_dim, num_heads)(x)

decoder_inputs = keras.Input(shape=(None,), dtype="int64", name="portugues")
x = PositionalEmbedding(sequence_length, vocab_size, embed_dim)(decoder_inputs)
x = TransformerDecoder(embed_dim, dense_dim, num_heads)(x, encoder_outputs)
x = layers.Dropout(0.7)(x)
decoder_outputs = layers.Dense(vocab_size, activation="softmax")(x)
transformer = keras.Model([encoder_inputs, decoder_inputs], decoder_outputs)

transformer.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"])

with tf.device('/device:GPU:0'):
    transformer.fit(train_ds, epochs=50, validation_data=val_ds)

Epoch 1/50
405/405 ━━━━━━━━━━━━━━━━━━━━ 101s 199ms/step - accuracy: 0.3568 - loss: 4.8937 - val_accuracy: 0.4089 - val_loss: 3.7776
Epoch 2/50
405/405 ━━━━━━━━━━━━━━━━━━━━ 116s 162ms/step - accuracy: 0.4199 - loss: 3.7086 - val_accuracy: 0.4726 - val_loss: 3.1763
Epoch 3/50
405/405 ━━━━━━━━━━━━━━━━━━━━ 82s 161ms/step - accuracy: 0.4659 - loss: 3.2567 - val_accuracy: 0.4944 - val_loss: 2.9916
Epoch 4/50
405/405 ━━━━━━━━━━━━━━━━━━━━ 81s 159ms/step - accuracy: 0.4887 - loss: 3.0552 - val_accuracy: 0.5126 - val_loss: 2.8426
Epoch 5/50
405/405 ━━━━━━━━━━━━━━━━━━━━ 82s 159ms/step - accuracy: 0.5112 - loss: 2.8758 - val_accuracy: 0.5458 - val_loss: 2.6399
Epoch 6/50
405/405 ━━━━━━━━━━━━━━━━━━━━ 63s 155ms/step - accuracy: 0.5417 - loss: 2.6627 - val_accuracy: 0.5782 - val_loss: 2.4369
Epoch 7/50
405/405 ━━━━━━━━━━━━━━━━━━━━ 84s 159ms/step - accuracy: 0.5795 - loss: 2.4162 - val_accuracy: 0.6155 - val_loss: 2.2133
Epoch 8/50
405/405 ━━━━━━━━━━━━━━━━━━━━ 63s 155ms/step - accuracy: 0.6142 - loss:

In [ ]:
# Salvando para uso posterior
transformer.save('../modelos/20240903_modelo_transformer_tradutor_50epochs.keras')

In [ ]:
import numpy as np

pt_vocab = target_vectorization.get_vocabulary()
pt_index_lookup = dict(zip(range(len(pt_vocab)), pt_vocab))
max_decoded_sentence_length = 40

def decode_sequence(input_sentence):
    tokenized_input_sentence = source_vectorization([input_sentence])
    decoded_sentence = "[start]"

    for i in range(max_decoded_sentence_length):
        tokenized_target_sentence = target_vectorization(
            [decoded_sentence])[:, :-1]
        predictions = transformer([tokenized_input_sentence, tokenized_target_sentence])
        sampled_token_index = np.argmax(predictions[0, i, :])
        sampled_token = pt_index_lookup[sampled_token_index]
        decoded_sentence += " " + sampled_token

        if sampled_token == "[end]":
            break

    return decoded_sentence

In [ ]:
test_eng_texts = [pair[0] for pair in test_pairs]
for _ in range(40):
    input_sentence = random.choice(test_eng_texts)
    print("-")
    print(input_sentence)
    print(decode_sequence(input_sentence))

-
No matter what mood your in, you can find the perfect song on Spotify.
[start] não importa qual o meu humor você pode encontrar a música perfeita no spotify [end]
-
Normally I enjoy Spotify, but recently it has been randomly closing on me while I'm using it. The service is great, but the app itself is not working well.
[start] normalmente gosto do spotify mas ultimamente ele tem sido forçado aleatoriamente me fechar enquanto estou usando o serviço é ótimo mas do aplicativo de si não está funcionando bem [end]
-
This app deserves more than 5 for me, all my old favourites came to bare. Great app it is...
[start] este aplicativo merece mais de 5 porque me forçou a usar [end]
-
In general the app is good, but recently when I play music it will occasionally not show the notification to show what song is playing or the app will freeze, I also can't unhide songs. Please fix these issues.
[start] no geral o aplicativo é bom mas recentemente quando toco música ele fica tocando ocasionalmente 

In [ ]:
#comentarios:
# REVIEW INVENTADA: 'Spotify has an amazing music selection, but the constant ads and limited skips make it frustrating to use without premium'
# REVIEW INVENTADA: 'Great audio quality, but I keep losing control over the volume when casting to my smart speaker. Is really inconvenient!'
# FRASE ALEATÓRIA QUE NÃO TEM SENTIDO : 'I love to eat Pizza, when is cold I drink wine'
# FRASE DA BASE TREINO: 'Unable to login .After this new update it shows error everytime I try to login

In [ ]:
reviews = ['Spotify has an amazing music selection, but the constant ads and limited skips make it frustrating to use without premium',
           'Great audio quality, but I keep losing control over the volume when casting to my smart speaker. Is really inconvenient!',
           'I love to eat pasta, when is cold I drink wine.',
           'Unable to login .After this new update it shows error everytime I try to login']

# Aplique a função decode_sequence para cada frase
for review in reviews:
    result = decode_sequence(review)
    print(result)

[start] o spotify tem uma seleção musical incrível mas os anúncios são péssimos e pula repete as pulos para usar sem premium [end]
[start] Ótima qualidade de áudio mas continuo perdendo a controle via volume ao transmitir para meus fones de ouvido é realmente inconveniente [end]
[start] adoro chorar a conclusão é que quando eu aprimoro a músicasplaylists que [end]
[start] não consigo fazer login após a nova atualização ele mostra que o erro toda vez que tento fazer login [end]


# Modelo com ajustes - 100% dos dados

ajustes da última versão

embed_dim = 512
adicionar o early_stop

In [10]:
import tensorflow as tf
import pandas as pd
from tensorflow.keras.layers import TextVectorization
from tensorflow.keras.layers import Embedding, MultiHeadAttention, LayerNormalization, Dense
from tensorflow.keras.models import Model

data = pd.read_csv('../data/df_traduzido.csv')
data.head(3)

,Time_submitted,Review,Rating,Total_thumbsup,Reply,Review_Traduzida,Review_trazudida
0,2022-07-09 15:00:00,"Great music service, the audio is high quality...",5,2,NaN,"Ótimo serviço de música, o áudio é de alta qua...","Ótimo serviço de música, o áudio é de alta qua..."
1,2022-07-09 14:21:22,Please ignore previous negative rating. This a...,5,1,NaN,"Por favor, ignore a avaliação negativa anterio...","Por favor, ignore a avaliação negativa anterio..."
2,2022-07-09 13:27:32,"This pop-up ""Get the best Spotify experience o...",4,0,NaN,Este pop-up &quot;Obtenha a melhor experiência...,Este pop-up &quot;Obtenha a melhor experiência...


In [11]:
data.drop(['Reply', 'Review_Traduzida'], axis=1, inplace=True)

In [12]:
data.rename(columns={'Review_trazudida': 'Review_Traduzida'}, inplace=True)

In [13]:
data.head(5)

,Time_submitted,Review,Rating,Total_thumbsup,Review_Traduzida
0,2022-07-09 15:00:00,"Great music service, the audio is high quality...",5,2,"Ótimo serviço de música, o áudio é de alta qua..."
1,2022-07-09 14:21:22,Please ignore previous negative rating. This a...,5,1,"Por favor, ignore a avaliação negativa anterio..."
2,2022-07-09 13:27:32,"This pop-up ""Get the best Spotify experience o...",4,0,Este pop-up &quot;Obtenha a melhor experiência...
3,2022-07-09 13:26:45,Really buggy and terrible to use as of recently,1,1,Realmente cheio de bugs e terrível de usar ult...
4,2022-07-09 13:20:49,Dear Spotify why do I get songs that I didn't ...,1,1,"Caro Spotify, por que recebo músicas que não c..."


In [14]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

class TransformerEncoder(layers.Layer):
    def __init__(self, embed_dim, dense_dim, num_heads, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.dense_dim = dense_dim
        self.num_heads = num_heads
        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim)
        self.dense_proj = keras.Sequential(
            [layers.Dense(dense_dim, activation="relu"),
             layers.Dense(embed_dim),]
        )
        self.layernorm_1 = layers.LayerNormalization()
        self.layernorm_2 = layers.LayerNormalization()

    def call(self, inputs, mask=None):
        if mask is not None:
            mask = mask[:, tf.newaxis, :]
        attention_output = self.attention(
            inputs, inputs, attention_mask=mask)
        proj_input = self.layernorm_1(inputs + attention_output)
        proj_output = self.dense_proj(proj_input)
        return self.layernorm_2(proj_input + proj_output)

    def get_config(self):
        config = super().get_config()
        config.update({
            "embed_dim": self.embed_dim,
            "num_heads": self.num_heads,
            "dense_dim": self.dense_dim,
        })
        return config

In [15]:
print(data.columns)

Index(['Time_submitted', 'Review', 'Rating', 'Total_thumbsup',
       'Review_Traduzida'],
      dtype='object')


In [16]:
text_pairs = []

for line in data.itertuples(index=False):
    english = str(line.Review)  # Converter para string
    portugues = str(line.Review_Traduzida)  # Converter para string

    # Tratar valores ausentes ou NaN, substituindo por uma string vazia
    if pd.isna(portugues):
        portugues = ""

    portugues = "[start] " + portugues + " [end]"

    text_pairs.append((english, portugues))

In [17]:
import random
print(random.choice(text_pairs))

random.shuffle(text_pairs)

num_val_samples = int(0.15 * len(text_pairs))
num_train_samples = len(text_pairs) - 2 * num_val_samples

train_pairs = text_pairs[:num_train_samples]
val_pairs = text_pairs[num_train_samples:num_train_samples + num_val_samples]
test_pairs = text_pairs[num_train_samples + num_val_samples:]

("Let me disable podcasts on this app that I pay money for, I am NEVER going to listen to them yet they cover half the screen when I open the app. Why is something I've never so much as clicked on the second listing on the home page, right in the center of the screen? Edit 2/3/22 still annoyed by podcasts being in my way", '[start] Deixe-me desabilitar podcasts neste aplicativo pelo qual pago, NUNCA vou ouvi-los, mas eles cobrem metade da tela quando abro o aplicativo. Por que algo em que nunca cliquei está na segunda listagem na página inicial, bem no centro da tela? Editar 2/3/22 ainda estou incomodado com os podcasts que estão no meu caminho [end]')


In [18]:
type(train_pairs)

list

In [19]:
import string
import re

strip_chars = string.punctuation + "¿"
strip_chars = strip_chars.replace("[", "")
strip_chars = strip_chars.replace("]", "")

def custom_standardization(input_string):
    lowercase = tf.strings.lower(input_string)
    return tf.strings.regex_replace(
        lowercase, f"[{re.escape(strip_chars)}]", "")

vocab_size = 30000
sequence_length = 40

source_vectorization = layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length,
)
target_vectorization = layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length + 1,
    standardize=custom_standardization,
)

train_english_texts = [pair[0] for pair in train_pairs]
train_spanish_texts = [pair[1] for pair in train_pairs]
source_vectorization.adapt(train_english_texts)
target_vectorization.adapt(train_spanish_texts)

In [20]:
batch_size = 64

def format_dataset(eng, pt):
    eng = source_vectorization(eng)
    pt = target_vectorization(pt)
    return ({
        "english": eng,
        "portugues": pt[:, :-1],
    }, pt[:, 1:])

def make_dataset(pairs):
    eng_texts, pt_texts = zip(*pairs)
    eng_texts = list(eng_texts)
    pt_texts = list(pt_texts)
    dataset = tf.data.Dataset.from_tensor_slices((eng_texts, pt_texts))
    dataset = dataset.batch(batch_size)
    dataset = dataset.map(format_dataset, num_parallel_calls=4)
    return dataset.shuffle(2048).prefetch(16).cache()

train_ds = make_dataset(train_pairs)
val_ds = make_dataset(val_pairs)

In [21]:
for inputs, targets in train_ds.take(1):
    print(f"inputs['english'].shape: {inputs['english'].shape}")
    print(f"inputs['portugues'].shape: {inputs['portugues'].shape}")
    print(f"targets.shape: {targets.shape}")

inputs['english'].shape: (64, 40)
inputs['portugues'].shape: (64, 40)
targets.shape: (64, 40)


In [22]:
class TransformerDecoder(layers.Layer):
    def __init__(self, embed_dim, dense_dim, num_heads, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.dense_dim = dense_dim
        self.num_heads = num_heads
        self.attention_1 = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim)
        self.attention_2 = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim)
        self.dense_proj = keras.Sequential(
            [layers.Dense(dense_dim, activation="relu"),
             layers.Dense(embed_dim),]
        )
        self.layernorm_1 = layers.LayerNormalization()
        self.layernorm_2 = layers.LayerNormalization()
        self.layernorm_3 = layers.LayerNormalization()
        self.supports_masking = True

    def get_causal_attention_mask(self, inputs):
        input_shape = tf.shape(inputs)
        batch_size, sequence_length = input_shape[0], input_shape[1]
        i = tf.range(sequence_length)[:, tf.newaxis]
        j = tf.range(sequence_length)
        mask = tf.cast(i >= j, dtype="int32")
        mask = tf.reshape(mask, (1, input_shape[1], input_shape[1]))
        mult = tf.concat(
            [tf.expand_dims(batch_size, -1),
             tf.constant([1, 1], dtype=tf.int32)], axis=0)
        return tf.tile(mask, mult)

    def call(self, inputs, encoder_outputs, mask=None):
        causal_mask = self.get_causal_attention_mask(inputs)
        if mask is not None:
            padding_mask = tf.cast(
                mask[:, tf.newaxis, :], dtype="int32")
            padding_mask = tf.minimum(padding_mask, causal_mask)
        else:
            padding_mask = mask
        attention_output_1 = self.attention_1(
            query=inputs,
            value=inputs,
            key=inputs,
            attention_mask=causal_mask)
        attention_output_1 = self.layernorm_1(inputs + attention_output_1)
        attention_output_2 = self.attention_2(
            query=attention_output_1,
            value=encoder_outputs,
            key=encoder_outputs,
            attention_mask=padding_mask,
        )
        attention_output_2 = self.layernorm_2(
            attention_output_1 + attention_output_2)
        proj_output = self.dense_proj(attention_output_2)
        return self.layernorm_3(attention_output_2 + proj_output)

class PositionalEmbedding(layers.Layer):
    def __init__(self, sequence_length, input_dim, output_dim, **kwargs):
        super().__init__(**kwargs)
        self.token_embeddings = layers.Embedding(
            input_dim=input_dim, output_dim=output_dim)
        self.position_embeddings = layers.Embedding(
            input_dim=sequence_length, output_dim=output_dim)
        self.sequence_length = sequence_length
        self.input_dim = input_dim
        self.output_dim = output_dim

    def call(self, inputs):
        length = tf.shape(inputs)[-1]
        positions = tf.range(start=0, limit=length, delta=1)
        embedded_tokens = self.token_embeddings(inputs)
        embedded_positions = self.position_embeddings(positions)
        return embedded_tokens + embedded_positions

    def get_config(self):
        config = super(PositionalEmbedding, self).get_config()
        config.update({
            "output_dim": self.output_dim,
            "sequence_length": self.sequence_length,
            "input_dim": self.input_dim,
        })
        return config

In [23]:
from tensorflow.keras.callbacks import EarlyStopping

embed_dim = 512
dense_dim = 4096
num_heads = 12

encoder_inputs = keras.Input(shape=(None,), dtype="int64", name="english")
x = PositionalEmbedding(sequence_length, vocab_size, embed_dim)(encoder_inputs)
encoder_outputs = TransformerEncoder(embed_dim, dense_dim, num_heads)(x)

decoder_inputs = keras.Input(shape=(None,), dtype="int64", name="portugues")
x = PositionalEmbedding(sequence_length, vocab_size, embed_dim)(decoder_inputs)
x = TransformerDecoder(embed_dim, dense_dim, num_heads)(x, encoder_outputs)
x = layers.Dropout(0.7)(x)
decoder_outputs = layers.Dense(vocab_size, activation="softmax")(x)
transformer = keras.Model([encoder_inputs, decoder_inputs], decoder_outputs)

transformer.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"])

# Definindo EarlyStopping
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)
with tf.device('/device:GPU:0'):
    transformer.fit(train_ds, epochs=30, validation_data=val_ds)

Epoch 1/30
674/674 ━━━━━━━━━━━━━━━━━━━━ 335s 462ms/step - accuracy: 0.3760 - loss: 4.3825 - val_accuracy: 0.4658 - val_loss: 3.2124
Epoch 2/30
674/674 ━━━━━━━━━━━━━━━━━━━━ 305s 452ms/step - accuracy: 0.4741 - loss: 3.1774 - val_accuracy: 0.5083 - val_loss: 2.8370
Epoch 3/30
674/674 ━━━━━━━━━━━━━━━━━━━━ 304s 451ms/step - accuracy: 0.5109 - loss: 2.8413 - val_accuracy: 0.5404 - val_loss: 2.5869
Epoch 4/30
674/674 ━━━━━━━━━━━━━━━━━━━━ 305s 453ms/step - accuracy: 0.5434 - loss: 2.5856 - val_accuracy: 0.5673 - val_loss: 2.4180
Epoch 5/30
674/674 ━━━━━━━━━━━━━━━━━━━━ 305s 453ms/step - accuracy: 0.5729 - loss: 2.3827 - val_accuracy: 0.5905 - val_loss: 2.2765
Epoch 6/30
674/674 ━━━━━━━━━━━━━━━━━━━━ 322s 453ms/step - accuracy: 0.6013 - loss: 2.1947 - val_accuracy: 0.6209 - val_loss: 2.1182
Epoch 7/30
674/674 ━━━━━━━━━━━━━━━━━━━━ 305s 452ms/step - accuracy: 0.6290 - loss: 2.0240 - val_accuracy: 0.6447 - val_loss: 1.9823
Epoch 8/30
674/674 ━━━━━━━━━━━━━━━━━━━━ 306s 454ms/step - accuracy: 0.6538 -

In [28]:
# Salvando para uso posterior
transformer.save('../data/20240903_modelo_transformer_tradutor_30epochs.keras')

In [25]:
import numpy as np

pt_vocab = target_vectorization.get_vocabulary()
pt_index_lookup = dict(zip(range(len(pt_vocab)), pt_vocab))
max_decoded_sentence_length = 40

def decode_sequence(input_sentence):
    tokenized_input_sentence = source_vectorization([input_sentence])
    decoded_sentence = "[start]"

    for i in range(max_decoded_sentence_length):
        tokenized_target_sentence = target_vectorization(
            [decoded_sentence])[:, :-1]
        predictions = transformer([tokenized_input_sentence, tokenized_target_sentence])
        sampled_token_index = np.argmax(predictions[0, i, :])
        sampled_token = pt_index_lookup[sampled_token_index]
        decoded_sentence += " " + sampled_token

        if sampled_token == "[end]":
            break

    return decoded_sentence

In [26]:
test_eng_texts = [pair[0] for pair in test_pairs]
for _ in range(40):
    input_sentence = random.choice(test_eng_texts)
    print("-")
    print(input_sentence)
    print(decode_sequence(input_sentence))

-
Ever since the February 6 update the app has lost all car bluetooth functionality. No car controls, no song display and only occasionally will music play through the cars speakers. Getting real sick of paying for a premium service and having the developers not even bother fixing the problems. Do better Spotify.
[start] desde a primeiros 6 minutos o aplicativo perdeu o controle de transmissão a funcionalidade do car bluetooth não há controles de música para inesperadamente e apenas a música que toca no carro os dados ficam visíveis em um serviço premium
-
My downloads often get removed when I turn on my phone. Sometimes the playlist skips songs.
[start] meus downloads geralmente são removidas quando fecho quando ligo meu telefone Às vezes a playlist pula as músicas [end]
-
Really only good if you have money Listen, it's a good app when you actually have money. I'll give it that. But when you're unable to pay for a subscription things can get absolutely terrible. I'd be fine with like 

In [ ]:
#comentarios:
# REVIEW INVENTADA: 'Spotify has an amazing music selection, but the constant ads and limited skips make it frustrating to use without premium'
# REVIEW INVENTADA: 'Great audio quality, but I keep losing control over the volume when casting to my smart speaker. Is really inconvenient!'
# FRASE ALEATÓRIA QUE NÃO TEM SENTIDO : 'I love to eat Pizza, when is cold I drink wine'
# FRASE DA BASE TREINO: 'Unable to login .After this new update it shows error everytime I try to login

In [27]:
reviews = ['Spotify has an amazing music selection, but the constant ads and limited skips make it frustrating to use without premium',
           'Great audio quality, but I keep losing control over the volume when casting to my smart speaker. Is really inconvenient!',
           'I love to eat pasta, when is cold I drink wine.',
           'Unable to login .After this new update it shows error everytime I try to login']

# Aplique a função decode_sequence para cada frase
for review in reviews:
    result = decode_sequence(review)
    print(result)

[start] o spotify tem uma seleção de músicas incrível mas os anúncios constantes e anúncios constantes para fazer ele é frustrante usar sem premium [end]
[start] Ótima qualidade de áudio mas continuo perdendo perco o controle sobre o volume ao transmitir para meu altofalante inteligente é muito inconveniente [end]
[start] adoro como sofri sem dúvida quando estamos pedalando a bicicleta para [end]
[start] não é possível fazer login após essa nova atualização ele mostra toda vez que tento fazer login [end]
